# Vehicle Detection — Neural Network Training

In [1]:
import sys
import platform

print("Python executable:", sys.executable)
print("Python version:", platform.python_version())
print("Environment is ready!")

Python executable: c:\Users\User\jupyter\.venv\Scripts\python.exe
Python version: 3.11.4
Environment is ready!


## Dataset Acquisition
.

In [2]:
from roboflow import Roboflow
from getpass import getpass

api_key = getpass("Enter your Roboflow API Key: ")

rf = Roboflow(api_key=api_key)

project = rf.workspace("yolo-al3ff").project(
    "vehicle-detection-dataset-sgafv"
)

version = project.version(1)

dataset = version.download("yolov8")

print("Dataset downloaded successfully!")
print("Dataset location:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...


KeyboardInterrupt: 

In [ ]:
from pathlib import Path

dataset_path = Path(dataset.location)

print("Dataset folders:")

for item in dataset_path.iterdir():
    print(item.name)

data_yaml = dataset_path / "data.yaml"

print("\nConfiguration exists:", data_yaml.exists())

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

print("YOLO11 Nano model loaded!")

In [ ]:
train_results = model.train(
    data=str(data_yaml),

    epochs=5,
    imgsz=416,
    batch=4,

    device="cpu",
    workers=0,

    optimizer="auto",
    pretrained=True,

    project="runs/vehicle_detection",
    name="experiment_1_baseline",

    plots=True,
    save=True,
    exist_ok=True
)

print("\nTraining completed!")
print(train_results.results_dict)

In [ ]:
best_model = YOLO(
    "runs/vehicle_detection/experiment_1_baseline/weights/best.pt"
)

metrics = best_model.val(
    data=str(data_yaml),
    device="cpu"
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## Experiment 1 — Baseline YOLO11n

- Model: YOLO11n
- Epochs: 5
- Image size: 416
- Batch size: 4
- Device: CPU
- Dataset: Vehicle Detection Dataset

In [ ]:


model = YOLO("yolo11n.pt")

train_results = model.train(
    data=str(data_yaml),
    epochs=5,
    imgsz=416,
    batch=4,
    device="cpu",
    workers=0,
    optimizer="auto",
    pretrained=True,
    project="runs/vehicle_detection",
    name="experiment_1_baseline",
    plots=True,
    save=True,
    exist_ok=True
)

print("Baseline training completed!")
print(train_results.results_dict)

## Experiment 2 - more baselines

In [ ]:


model_2 = YOLO("yolo11n.pt")

train_results_2 = model_2.train(
    data=str(data_yaml),
    epochs=10,
    imgsz=416,
    batch=4,
    device="cpu",
    workers=0,
    optimizer="auto",
    pretrained=True,
    project="runs/vehicle_detection",
    name="experiment_2_extended",
    plots=True,
    save=True,
    exist_ok=True
)

print("Experiment 2 completed!")
print(train_results_2.results_dict)

In [ ]:

import pandas as pd
from pathlib import Path

# Load both trained models
model_1 = YOLO(
    "runs/vehicle_detection/experiment_1_baseline/weights/best.pt"
)

model_2 = YOLO(
    "runs/vehicle_detection/experiment_2_extended/weights/best.pt"
)

# Determine which dataset split to evaluate
test_images = dataset_path / "test" / "images"
split_name = "test" if test_images.exists() else "val"

print("Evaluation split:", split_name)

# Evaluate both models on the same split
metrics_1 = model_1.val(
    data=str(data_yaml),
    split=split_name,
    device="cpu",
    workers=0
)

metrics_2 = model_2.val(
    data=str(data_yaml),
    split=split_name,
    device="cpu",
    workers=0
)

# Build comparison table
comparison = pd.DataFrame([
    {
        "Experiment": "Baseline (5 epochs)",
        "Precision": metrics_1.box.mp,
        "Recall": metrics_1.box.mr,
        "mAP50": metrics_1.box.map50,
        "mAP50-95": metrics_1.box.map
    },
    {
        "Experiment": "Extended (10 epochs)",
        "Precision": metrics_2.box.mp,
        "Recall": metrics_2.box.mr,
        "mAP50": metrics_2.box.map50,
        "mAP50-95": metrics_2.box.map
    }
])

display(comparison.round(4))

comparison.to_csv(
    "experiment_comparison.csv",
    index=False
)

print("Comparison saved!")

In [ ]:
results = model_2.predict(
    source=str(test_images if test_images.exists() else dataset_path / "valid" / "images"),
    conf=0.25,
    save=True,
    device="cpu",
    project="runs/vehicle_detection",
    name="final_predictions",
    exist_ok=True
)

print("Predictions saved!")